# 🛠️ Fase 2: Limpieza de Datos y Feature Engineering

En este notebook aplicaremos las transformaciones necesarias identificadas durante el EDA para preparar el dataset para los modelos generativos y predictivos.

## Objetivos:
1. **Limpieza**: Filtrar registros inválidos (fallecidos) y eliminar columnas con exceso de nulos.
2. **Ingeniería de Variables**: Agrupar diagnósticos ICD-9 y crear métricas agregadas.
3. **Imputación**: Manejar los valores faltantes en variables categóricas.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

## 1. Carga de Datos y Limpieza Inicial

In [2]:
df = pd.read_csv("../data/diabetic_data.csv")
df.replace('?', np.nan, inplace=True)

print(f"Forma original: {df.shape}")

Forma original: (101766, 50)


### 1.1. Filtrado de Registros Inválidos
Eliminamos pacientes fallecidos o en cuidados paliativos (`discharge_disposition_id` en [11, 13, 14, 19, 20, 21]).

In [3]:
death_ids = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(death_ids)]

print(f"Forma tras eliminar fallecidos: {df.shape}")

Forma tras eliminar fallecidos: (99343, 50)


### 1.2. Limpieza de Género
Eliminamos los escasos registros con género desconocido.

In [4]:
df = df[df['gender'] != 'Unknown/Invalid']
print(f"Forma tras limpiar género: {df.shape}")

Forma tras limpiar género: (99340, 50)


### 1.3. Eliminación de Columnas con Exceso de Nulos o Baja Relevancia
Eliminamos `weight` (97% nulos), `payer_code` (40% nulos y baja relevancia clínica) y medicamentos con varianza nula.

In [5]:
cols_to_drop = ['weight', 'payer_code', 'encounter_id', 'patient_nbr']
df.drop(columns=cols_to_drop, inplace=True)

# Medicamentos con muy poca variabilidad detectados en EDA
low_variance_meds = [
    'acetohexamide', 'troglitazone', 'examide', 'citoglipton', 
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]
df.drop(columns=low_variance_meds, inplace=True)

print(f"Forma tras eliminar columnas: {df.shape}")

Forma tras eliminar columnas: (99340, 39)


## 2. Ingeniería de Variables

### 2.1. Agrupación de Diagnósticos (ICD-9)
Mapeamos los códigos de diagnóstico a categorías clínicas simplificadas para reducir la cardinalidad.

In [6]:
def map_icd9(code):
    if pd.isnull(code):
        return 'None'
    
    # Códigos V y E -> Other
    if code.startswith(('V', 'E')):
        return 'Other'
    
    try:
        # Algunos códigos pueden tener caracteres extraños o ser flotantes con puntos
        f_code = float(code)
        if 390 <= f_code <= 459 or int(f_code) == 785:
            return 'Circulatory'
        elif 460 <= f_code <= 519 or int(f_code) == 786:
            return 'Respiratory'
        elif 520 <= f_code <= 579 or int(f_code) == 787:
            return 'Digestive'
        elif int(f_code) == 250:
            return 'Diabetes'
        elif 800 <= f_code <= 999:
            return 'Injury'
        elif 710 <= f_code <= 739:
            return 'Musculoskeletal'
        elif 580 <= f_code <= 629 or int(f_code) == 788:
            return 'Genitourinary'
        elif 140 <= f_code <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except ValueError:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(map_icd9)

print("Conteo de categorías en diag_1:")
print(df['diag_1'].value_counts())

Conteo de categorías en diag_1:
diag_1
Circulatory        29680
Other              17793
Respiratory        13934
Digestive           9333
Diabetes            8661
Injury              6851
Genitourinary       5002
Musculoskeletal     4935
Neoplasms           3131
None                  20
Name: count, dtype: int64


### 2.2. Creación de Nuevas Métricas
- **prior_visits**: Suma de visitas previas.
- **any_med_change**: Flag binario si hubo cambios en la medicación.

In [7]:
df['prior_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']

df['any_med_change'] = df['change'].apply(lambda x: 1 if x == 'Ch' else 0)

print(f"Estadísticas de prior_visits: {df['prior_visits'].describe()}")

Estadísticas de prior_visits: count    99340.000000
mean         1.198661
std          2.294240
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max         80.000000
Name: prior_visits, dtype: float64


## 3. Imputación de Nulos

In [8]:
# Imputación de raza con la moda
df['race'].fillna(df['race'].mode()[0], inplace=True)

# Imputación de especialidad médica como 'Missing'
df['medical_specialty'].fillna('Missing', inplace=True)

print("Nulos por columna tras imputación:")
print(df.isnull().sum().sum())

Nulos por columna tras imputación:
176694


/tmp/ipykernel_113/1984727172.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['race'].fillna(df['race'].mode()[0], inplace=True)
/tmp/ipykernel_113/1984727172.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', t

## 4. Guardar Dataset Limpio

In [9]:
df.to_csv("../data/diabetic_data_clean.csv", index=False)
print("Dataset guardado como 'diabetic_data_clean.csv'")

Dataset guardado como 'diabetic_data_clean.csv'
